In [1]:
#класстеризация autoencoder

from __future__ import annotations
import os, glob, sys, argparse
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler 

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def parse_args():
    ap = argparse.ArgumentParser("Clustering only for tickers that start at the earliest date (full period, GPU-ready).")
    ap.add_argument("--data-dir", default="data", help="Каталог с CSV (если нет --all-csv)")
    ap.add_argument("--all-csv", default="data/prices_all.csv", help="Общий CSV (tidy)")
    ap.add_argument("--price-col", default="auto", help="auto | Adj Close | Close | Open | High | Low")
    ap.add_argument("--min-rows", type=int, default=50, help="Мин. наблюдений на тикер (жёсткий отсев)")
    ap.add_argument("--algo", nargs="+", default=["agglo","kmeans"], choices=["agglo","kmeans"], help="Алгоритмы")
    ap.add_argument("--n-clusters", nargs="+", type=int, default=[6,8,10], help="Список K")
    ap.add_argument("--latent-dim", nargs="+", type=int, default=[2,3], help="Размер z-представления AE")
    ap.add_argument("--hidden1", type=int, default=256)
    ap.add_argument("--hidden2", type=int, default=256)
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--runs", type=int, default=6)
    ap.add_argument("--batch-size", type=int, default=16)  # не увеличиваем
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--pdf-file", default="clusters_fullstart_report.pdf")
    ap.add_argument("--out-csv", default="cluster_fullstart_assignments.csv")
    ap.add_argument("--out-html", default="cluster_fullstart_table.html")
    args, _unk = ap.parse_known_args(sys.argv[1:])
    return args

# ---------------- helpers ----------------
def safe_float(x: Any) -> float:
    """Безопасное приведение к float: поддерживает numpy-скаляры и массивы (берём .item() или mean)."""
    try:
        # numpy scalars имеют .item()
        item = getattr(x, "item", None)
        if callable(item):
            return float(item())
    except Exception:
        pass
    try:
        arr = np.asarray(x)
        if arr.size == 1:
            return float(arr.reshape(-1)[0])
        return float(np.mean(arr))
    except Exception:
        return float(x)

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Apple Silicon (macOS)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_prices(path_all_csv: str, data_dir: str) -> pd.DataFrame:
    if os.path.exists(path_all_csv):
        df = pd.read_csv(path_all_csv)
    else:
        parts = []
        for p in glob.glob(os.path.join(data_dir, "*.csv")):
            try:
                dfi = pd.read_csv(p)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit(f"Нет CSV с колонкой 'Ticker' в {data_dir}")
        df = pd.concat(parts, ignore_index=True)

    ren = {}
    for want in ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]:
        for c in df.columns:
            if c.strip().lower() == want.lower():
                ren[c] = want
                break
    df = df.rename(columns=ren)
    if "date" not in df.columns or "Ticker" not in df.columns:
        raise SystemExit("Нужны колонки: date, Ticker (+ цены).")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date","Ticker"]).reset_index(drop=True)
    return df

def pick_price_col(df: pd.DataFrame, pref: str) -> str:
    if pref != "auto":
        if pref in df.columns:
            return pref
        raise SystemExit(f"--price-col={pref} не найден.")
    return "Adj Close" if "Adj Close" in df.columns else ("Close" if "Close" in df.columns else "Close")

def filter_tickers_start_at_global_min(df: pd.DataFrame) -> List[str]:
    gmin = df["date"].min()
    firsts = df.groupby("Ticker")["date"].min()
    keep = firsts[firsts == gmin].index.tolist()
    return keep

def build_matrix_full(df: pd.DataFrame, price_col: str, min_rows: int,
                      tickers_keep: List[str]) -> Tuple[np.ndarray, List[str], pd.DatetimeIndex, pd.DataFrame]:
    dff = df[df["Ticker"].isin(set(tickers_keep))].copy()
    wide = dff.pivot(index="date", columns="Ticker", values=price_col).sort_index()

    good_cols = [c for c in wide.columns if wide[c].notna().sum() >= min_rows]
    wide = wide[good_cols]
    if wide.shape[1] == 0:
        raise SystemExit("После фильтрации по min_rows не осталось тикеров.")

    # требуем отсутствие NaN (по условию данные нормальные)
    if wide.isna().any().any():
        bad = wide.columns[wide.isna().any()].tolist()
        raise SystemExit(f"Есть пропуски (NaN) у: {bad[:10]}{' ...' if len(bad)>10 else ''}. Почините данные или уберите тикеры.")

    wmin = wide.min(axis=0); wmax = wide.max(axis=0)
    denom = (wmax - wmin).replace(0, np.nan)
    scaled = (wide - wmin) / denom
    scaled = scaled.fillna(0.0)

    X = scaled.T.values.astype(np.float32)
    companies = list(scaled.columns)
    dates = scaled.index
    return X, companies, dates, scaled

class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, h1: int, h2: int, z: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Linear(h2, z), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z, h2), nn.ReLU(),
            nn.Linear(h2, h1), nn.ReLU(),
            nn.Linear(h1, input_dim), nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y

def train_ae(X: np.ndarray, zdim: int, h1: int, h2: int,
             epochs: int, batch: int, lr: float, seed: int,
             device: torch.device) -> Tuple[np.ndarray, float]:
    torch.manual_seed(seed); np.random.seed(seed)
    n, d = X.shape
    model = Autoencoder(d, h1, h2, zdim).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    X_tensor = torch.from_numpy(X)
    if device.type == "cuda":
        X_tensor = X_tensor.pin_memory()
        torch.backends.cudnn.benchmark = True

    loader = torch.utils.data.DataLoader(
        X_tensor, batch_size=batch, shuffle=True,
        pin_memory=(device.type=="cuda"), num_workers=0
    )

    scaler = GradScaler(device="cuda", enabled=(device.type=="cuda"))

    model.train()
    for i in range(epochs):
        for batch_x in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(device.type=="cuda")):
                recon = model(batch_x)
                loss = loss_fn(recon, batch_x)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

    model.eval()
    with torch.no_grad():
        Z_parts = []
        for batch_x in torch.utils.data.DataLoader(X_tensor, batch_size=batch, shuffle=False):
            batch_x = batch_x.to(device, non_blocking=True)
            z = model.encoder(batch_x).cpu().numpy()
            Z_parts.append(z)
        Z = np.vstack(Z_parts)

        recon_err = 0.0
        count = 0
        for batch_x in torch.utils.data.DataLoader(X_tensor, batch_size=batch, shuffle=False):
            bx = batch_x.to(device, non_blocking=True)
            r = model(bx).cpu().numpy()
            recon_err += float(np.sum((r - batch_x.numpy())**2))
            count += batch_x.shape[0]
        recon_mse = recon_err / (count * d)

    return Z, recon_mse

# ---------------- Clustering ----------------
def cluster_and_score(Z: np.ndarray, algo: str, k: int, seed: int) -> Dict:
    if algo == "agglo":
        labels = AgglomerativeClustering(n_clusters=k).fit_predict(Z)
    elif algo == "kmeans":
        labels = KMeans(n_clusters=k, random_state=seed, n_init="auto").fit_predict(Z)
    else:
        raise ValueError("algo must be 'agglo' or 'kmeans'")

    if len(set(labels)) > 1 and Z.shape[0] > k:
        sil = silhouette_score(Z, labels)
        cal = calinski_harabasz_score(Z, labels)
        dav = davies_bouldin_score(Z, labels)
    else:
        sil, cal, dav = -np.inf, -np.inf, np.inf

    return {
        "labels": labels,
        "silhouette": safe_float(sil),
        "calinski":  safe_float(cal),
        "davies":    safe_float(dav),
    }

def save_pdf(pdf_path: str, best: dict, companies: List[str], scaled: pd.DataFrame, price_col: str):
    Z = best["Z"]; labels = best["labels"]; algo = best["algo"]; K = best["n_clusters"]
    zdim = best["latent_dim"]; run = best["run"]; m = best["metrics"]
    with PdfPages(pdf_path) as pdf:
        # PCA scatter
        pca2 = PCA(n_components=2).fit_transform(Z)
        fig = plt.figure(figsize=(7.2, 6.2))
        sc = plt.scatter(pca2[:,0], pca2[:,1], c=labels, cmap="tab10", s=70, edgecolor="k", linewidths=0.3)
        plt.title(f"{algo.upper()}  K={K}  z={zdim}  run={run}  "
                  f"CH={m['calinski']:.1f}  Sil={m['silhouette']:.3f}  DB={m['davies']:.3f}  ({price_col})")
        plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True, alpha=.35); plt.colorbar(sc, label="cluster")
        pdf.savefig(fig); plt.close(fig)

        for cl in sorted(set(labels)):
            idx = [i for i, lab in enumerate(labels) if lab == cl]
            show = idx[:min(12, len(idx))]
            if not show: continue
            cols = [companies[i] for i in show]
            fig2 = plt.figure(figsize=(12, 5))
            plt.plot(scaled.index, scaled[cols].values, linewidth=1.1)
            plt.title(f"{algo.upper()}  Cluster {cl}  (first {len(cols)} of {len(idx)})")
            plt.xlabel("Date"); plt.ylabel("Normalised price (0..1)"); plt.grid(True, alpha=.35)
            plt.tight_layout(); pdf.savefig(fig2); plt.close(fig2)

        cluster_map = pd.DataFrame({"Company": companies, "Cluster": labels}).sort_values(["Cluster","Company"])
        fig3, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off"); ax.set_title("Company → Cluster", fontsize=14, pad=18)
        tbl = ax.table(cellText=cluster_map.values, colLabels=cluster_map.columns, loc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.15)
        pdf.savefig(fig3); plt.close(fig3)

def save_interactive_table_html(path: str, cluster_map: pd.DataFrame, info: Dict[str,str]):
    table_html = cluster_map.to_html(index=False, classes="display compact", table_id="clusters")
    meta = "".join(f"<li><b>{k}:</b> {v}</li>" for k,v in info.items())
    html = f"""<!doctype html>
<html><head>
<meta charset="utf-8"/>
<title>Cluster assignments (full-start tickers)</title>
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css"/>
<style>body{{font-family:Arial, sans-serif; margin:18px}}</style>
</head><body>
<h2>Cluster assignments (tickers starting at global first date)</h2>
<ul>{meta}</ul>
{table_html}
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
<script>
$(function(){{
  $('#clusters').DataTable({{pageLength:25, lengthMenu:[10,25,50,100,200], order:[[1,'asc']], stateSave:true}});
}});
</script>
</body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

def main():
    args = parse_args()

    device = pick_device()
    print("[INFO] Device:", device)
    if device.type == "cuda":
        print("[INFO] CUDA build:", torch.version.cuda)
        print("[INFO] GPU:", torch.cuda.get_device_name(0))

    df = load_prices(args.all_csv, args.data_dir)
    price_col = pick_price_col(df, args.price_col)
    keep = filter_tickers_start_at_global_min(df)
    if not keep:
        raise SystemExit("Не найдено тикеров, начинающихся с первой даты в данных.")

    X, companies, dates, scaled = build_matrix_full(df, price_col, args.min_rows, keep)
    print(f"[INFO] Using price column: {price_col}")
    print(f"[INFO] Global first date: {df['date'].min().date()}")
    print(f"[INFO] Tickers starting at first date: {len(companies)}")
    print(f"[INFO] Matrix: companies={len(companies)}, time_points={X.shape[1]} (full period)")

    results = []
    total = len(args.latent_dim)*args.runs*len(args.algo)*len(args.n_clusters)
    pbar = tqdm(total=total, desc="grid")
    for z in args.latent_dim:
        for run in range(args.runs):
            Z, recon_mse = train_ae(X, z, args.hidden1, args.hidden2, args.epochs, args.batch_size, args.lr, seed=run, device=device)
            for algo in args.algo:
                for k in args.n_clusters:
                    metrics = cluster_and_score(Z, algo, k, seed=run)
                    results.append({
                        "algo": algo, "n_clusters": k, "latent_dim": z, "run": run,
                        "Z": Z, "labels": metrics["labels"], "metrics": metrics, "recon_mse": recon_mse
                    })
                    pbar.update(1)
    pbar.close()

    best = max(results, key=lambda r: r["metrics"]["calinski"])
    print("\n=== ЛУЧШАЯ КОНФИГУРАЦИЯ (по Calinski-Harabasz) ===")
    print(f"algo={best['algo']}, K={best['n_clusters']}, z={best['latent_dim']}, run={best['run']}")
    for k, v in best["metrics"].items():
        print(f"{k}: {safe_float(v):.6f}")
    print(f"Recon MSE: {safe_float(best['recon_mse']):.6f}")

    cluster_map = (
        pd.DataFrame({"Company": companies, "Cluster": best["labels"]})
        .sort_values(["Cluster","Company"]).reset_index(drop=True)
    )
    cluster_map.to_csv(args.out_csv, index=False)
    print(f"[INFO] Saved cluster assignments → {args.out_csv}")

    save_pdf(args.pdf_file, best, companies, scaled, price_col)
    print(f"[INFO] PDF report saved → {args.pdf_file}")

    info = {
        "Algorithm": best["algo"].upper(),
        "Clusters (K)": str(best["n_clusters"]),
        "Latent dim": str(best["latent_dim"]),
        "Run (seed)": str(best["run"]),
        "Calinski-Harabasz": f"{safe_float(best['metrics']['calinski']):.4f}",
        "Silhouette": f"{safe_float(best['metrics']['silhouette']):.4f}",
        "Davies-Bouldin": f"{safe_float(best['metrics']['davies']):.4f}",
        "Reconstruction MSE": f"{safe_float(best['recon_mse']):.6f}",
        "Price column": price_col,
        "Companies": str(len(companies)),
        "Time points": str(X.shape[1]),
        "Global first date": str(df['date'].min().date()),
    }
    save_interactive_table_html(args.out_html, cluster_map, info)
    print(f"[INFO] Interactive table saved → {args.out_html}")

if __name__ == "__main__":
    main()


[INFO] Device: cuda
[INFO] CUDA build: 12.1
[INFO] GPU: NVIDIA GeForce RTX 2060
[INFO] Using price column: Close
[INFO] Global first date: 2008-01-02
[INFO] Tickers starting at first date: 159
[INFO] Matrix: companies=159, time_points=4483 (full period)


grid:   1%|▏         | 1/72 [00:09<11:32,  9.75s/it]C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
grid:   6%|▌         | 4/72 [00:12<02:52,  2.53s/it]C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (8). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
grid:  26%|██▋       | 19/72 [00:28<01:02,  1.17s/it]C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\


=== ЛУЧШАЯ КОНФИГУРАЦИЯ (по Calinski-Harabasz) ===
algo=agglo, K=10, z=2, run=1
labels: 3.830189
silhouette: 0.588913
calinski: 2132.863037
davies: 0.398081
Recon MSE: 0.007533
[INFO] Saved cluster assignments → cluster_fullstart_assignments.csv
[INFO] PDF report saved → clusters_fullstart_report.pdf
[INFO] Interactive table saved → cluster_fullstart_table.html


In [3]:
#интерактивные графики

from __future__ import annotations
import os
import glob
from typing import List, Dict, Tuple

import pandas as pd
import numpy as np

from dash import Dash, dcc, html, Input, Output, State, no_update, dash_table
import plotly.graph_objs as go
from plotly.subplots import make_subplots

DATA_DIR = "data"
ALL_CSV = os.path.join(DATA_DIR, "prices_all.csv")
CLUSTERS_CSV = "cluster_fullstart_assignments.csv"

def load_prices() -> pd.DataFrame:
    """Читает prices_all.csv или все CSV из ./data; приводит имена колонок к стандартным."""
    if os.path.exists(ALL_CSV):
        df = pd.read_csv(ALL_CSV)
    else:
        parts: List[pd.DataFrame] = []
        for path in glob.glob(os.path.join(DATA_DIR, "*.csv")):
            try:
                dfi = pd.read_csv(path)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit("Не найден ни один CSV с колонкой 'Ticker' в ./data/")
        df = pd.concat(parts, ignore_index=True)

    ren = {}
    for want in ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]:
        for c in df.columns:
            if c.strip().lower() == want.lower():
                ren[c] = want
                break
    df = df.rename(columns=ren)
    if "date" not in df.columns or "Ticker" not in df.columns:
        raise ValueError("В CSV должны быть как минимум колонки 'date' и 'Ticker'.")

    df["date"] = pd.to_datetime(df["date"])
    cols_keep = ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]
    present = [c for c in cols_keep if c in df.columns]
    df = df[present].sort_values(["Ticker","date"]).reset_index(drop=True)
    return df

def load_clusters() -> pd.DataFrame:
    """Читает cluster_fullstart_assignments.csv (Company, Cluster) и переименовывает Company->Ticker."""
    if not os.path.exists(CLUSTERS_CSV):
        raise SystemExit(f"Не найден файл с результатами кластеризации: {CLUSTERS_CSV}")
    cm = pd.read_csv(CLUSTERS_CSV)
    if "Ticker" not in cm.columns and "Company" in cm.columns:
        cm = cm.rename(columns={"Company": "Ticker"})
    if "Ticker" not in cm.columns or "Cluster" not in cm.columns:
        raise ValueError("В кластерном CSV ожидаются колонки: 'Ticker' и 'Cluster' (или 'Company').")
    try:
        cm["Cluster"] = cm["Cluster"].astype(int)
    except Exception:
        pass
    return cm[["Ticker","Cluster"]].sort_values(["Cluster","Ticker"]).reset_index(drop=True)

prices = load_prices()
clusters = load_clusters()

TICKERS = sorted(prices["Ticker"].unique())
DATE_MIN = prices["date"].min().date()
DATE_MAX = prices["date"].max().date()
PRICE_FIELDS = [c for c in ["Adj Close","Close","Open","High","Low"] if c in prices.columns]
CLUSTER_IDS = sorted(clusters["Cluster"].unique(), key=lambda x: (str(type(x)), x))

# быстрые мапы
CLUSTERS_BY_ID: Dict = {cid: clusters.loc[clusters["Cluster"] == cid, "Ticker"].tolist()
                        for cid in CLUSTER_IDS}


def index_to_100(s: pd.Series) -> pd.Series:
    """Индексируем ряд так, чтобы первая доступная точка была 100 (для наглядного сравнения профилей)."""
    s = s.astype(float)
    first = s.dropna().iloc[0] if s.dropna().size else np.nan
    if pd.isna(first) or first == 0:
        return s * np.nan
    return (s / first) * 100.0

def minmax_01(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn, mx = s.min(), s.max()
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return s * 0.0
    return (s - mn) / (mx - mn)

def make_cluster_overview_fig(df: pd.DataFrame,
                              cluster_ids: List,
                              price_field: str,
                              date_from: pd.Timestamp | None,
                              date_to: pd.Timestamp | None,
                              norm_mode: str = "index100") -> go.Figure:
    """
    Рисует все выбранные кластеры:
    - тонкие линии: отдельные акции кластера
    - жирная линия: медианный профиль кластера
    """
    mask = pd.Series(True, index=df.index)
    if date_from is not None:
        mask &= (df["date"] >= date_from)
    if date_to is not None:
        mask &= (df["date"] <= date_to)
    dff = df.loc[mask].copy()
    if dff.empty:
        fig = go.Figure()
        fig.update_layout(title="Нет данных в выбранном периоде")
        return fig

    wide = dff.pivot(index="date", columns="Ticker", values=price_field).sort_index()

    if norm_mode == "index100":
        wide = wide.apply(index_to_100, axis=0)
        y_title = f"{price_field} (index = 100 @ start)"
    else:
        wide = wide.apply(minmax_01, axis=0)
        y_title = f"{price_field} (min-max 0..1)"

    fig = go.Figure()
    for cid in cluster_ids:
        tickers_c = [t for t in CLUSTERS_BY_ID.get(cid, []) if t in wide.columns]
        if not tickers_c:
            continue
        Wc = wide[tickers_c].copy()

        for t in tickers_c:
            fig.add_trace(go.Scatter(
                x=Wc.index, y=Wc[t], mode="lines",
                name=f"{t} (cl {cid})",
                legendgroup=f"cluster_{cid}",
                line=dict(width=1),
                opacity=0.35,
                hovertemplate=f"Cluster: {cid}<br>Ticker: {t}<br>Date: %{ { 'x' } }<br>Value: %{ { 'y' } }<extra></extra>"
            ))

        median_series = Wc.median(axis=1, skipna=True)
        fig.add_trace(go.Scatter(
            x=median_series.index, y=median_series.values, mode="lines",
            name=f"Cluster {cid} — median",
            legendgroup=f"cluster_{cid}",
            line=dict(width=3),
            hovertemplate=f"Cluster: {cid}<br>Date: %{ { 'x' } }<br>Median: %{ { 'y' } }<extra></extra>"
        ))

    fig.update_layout(
        margin=dict(l=40, r=40, t=50, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x",
        xaxis=dict(title=None, showgrid=True),
        yaxis=dict(title=y_title, showgrid=True),
        title=f"Кластеры: {', '.join(map(str, cluster_ids))}"
    )
    return fig

def make_ticker_detail_fig(df_sub: pd.DataFrame, price_field: str) -> go.Figure:

    def hovertext(row) -> str:
        parts = [f"Date: {row['date'].date()}"]
        for col in ["Open","High","Low","Close","Adj Close","Volume"]:
            if col in df_sub.columns and pd.notna(row.get(col, np.nan)):
                val = row[col]
                if col == "Volume":
                    parts.append(f"{col}: {int(val):,}".replace(",", " "))
                else:
                    parts.append(f"{col}: {val}")
        return "<br>".join(parts)

    ht = df_sub.apply(hovertext, axis=1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_sub["date"], y=df_sub[price_field], mode="lines",
        name=price_field, hoverinfo="text", hovertext=ht
    ))
    if "MA" in df_sub.columns:
        fig.add_trace(go.Scatter(
            x=df_sub["date"], y=df_sub["MA"], mode="lines",
            name=f"MA({df_sub.attrs.get('ma_window','N')})",
            line=dict(dash="dot"), hoverinfo="skip"
        ))
    if "Volume" in df_sub.columns:
        fig.add_trace(go.Bar(
            x=df_sub["date"], y=df_sub["Volume"],
            name="Volume", yaxis="y2", opacity=0.3, hoverinfo="skip"
        ))

    fig.update_layout(
        margin=dict(l=40, r=40, t=50, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        xaxis=dict(title=None, showgrid=True),
        yaxis=dict(title=price_field, showgrid=True),
        yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False),
        title="Детальный график тикера"
    )
    return fig


app = Dash(__name__)
app.title = "Clusters & Stocks — Interactive"

# стартовые значения
default_cluster = CLUSTER_IDS[0]
default_ticker = CLUSTERS_BY_ID[default_cluster][0] if CLUSTERS_BY_ID[default_cluster] else TICKERS[0]

app.layout = html.Div([
    html.H2("Кластеры акций — интерактивный просмотр"),

    html.Div([
        html.Div([
            html.Label("Кластеры"),
            dcc.Dropdown(
                id="clusters-dd",
                options=[{"label": str(cid), "value": cid} for cid in CLUSTER_IDS],
                value=[default_cluster],
                multi=True,
                clearable=False,
                style={"minWidth":"240px"}
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Показатель цены"),
            dcc.Dropdown(
                id="price-field-dd",
                options=[{"label": f, "value": f} for f in PRICE_FIELDS],
                value=PRICE_FIELDS[0],
                clearable=False,
                style={"minWidth":"180px"}
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Нормализация"),
            dcc.Dropdown(
                id="norm-mode-dd",
                options=[
                    {"label":"Index-100 (от начала периода)", "value":"index100"},
                    {"label":"Min-Max 0..1", "value":"minmax"}
                ],
                value="index100", clearable=False, style={"minWidth":"230px"}
            )
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Период"),
            dcc.DatePickerRange(
                id="date-range",
                min_date_allowed=DATE_MIN,
                max_date_allowed=DATE_MAX,
                start_date=DATE_MIN,
                end_date=DATE_MAX,
                display_format="YYYY-MM-DD"
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

    ], style={"marginBottom":"12px"}),

    dcc.Graph(id="clusters-graph", style={"height":"55vh"}),

    html.Hr(),

    html.Div([
        html.Div([
            html.Label("Тикер"),
            dcc.Dropdown(
                id="ticker-dd",
                options=[{"label": t, "value": t} for t in TICKERS],
                value=default_ticker,
                clearable=False,
                style={"minWidth":"220px"}
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Скользящая средняя (дней)"),
            dcc.Input(id="ma-window", type="number", value=20, min=1, step=1, style={"width":"90px"})
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),
    ], style={"marginBottom":"12px"}),

    dcc.Graph(id="ticker-graph", style={"height":"45vh"}),

    html.Hr(),

    html.H3("Таблица участников кластеров"),
    dash_table.DataTable(
        id="clusters-table",
        columns=[
            {"name":"Ticker", "id":"Ticker"},
            {"name":"Cluster", "id":"Cluster"},
        ],
        data=clusters.to_dict("records"),
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_size=20,
        style_table={"overflowX":"auto"},
        style_cell={"fontFamily":"monospace", "fontSize":"14px", "padding":"6px"},
        style_header={"fontWeight":"bold"},
        row_selectable="single",
    ),
    html.Div(id="table-info", style={"marginTop":"6px", "color":"#555"})
], style={"padding":"16px"})

@app.callback(
    Output("clusters-graph", "figure"),
    Input("clusters-dd", "value"),
    Input("price-field-dd", "value"),
    Input("norm-mode-dd", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
)
def update_clusters_graph(cluster_ids, price_field, norm_mode, start_date, end_date):
    if not cluster_ids or not price_field:
        return go.Figure()
    fig = make_cluster_overview_fig(
        prices[["date","Ticker",price_field,"Volume"]].copy(),
        cluster_ids=cluster_ids,
        price_field=price_field,
        date_from=pd.to_datetime(start_date) if start_date else None,
        date_to=pd.to_datetime(end_date) if end_date else None,
        norm_mode=norm_mode
    )
    return fig

@app.callback(
    Output("ticker-dd", "options"),
    Output("ticker-dd", "value"),
    Input("clusters-dd", "value"),
    State("ticker-dd", "value"),
)
def sync_ticker_list(cluster_ids, current_ticker):
    """Сужаем список тикеров к выбранным кластерам; сохраняем текущий, если он входит."""
    if not cluster_ids:
        opts = [{"label": t, "value": t} for t in TICKERS]
        return opts, current_ticker or TICKERS[0]
    tickers_allowed = sorted({t for cid in cluster_ids for t in CLUSTERS_BY_ID.get(cid, [])})
    if not tickers_allowed:
        opts = [{"label": t, "value": t} for t in TICKERS]
        return opts, current_ticker or TICKERS[0]
    opts = [{"label": t, "value": t} for t in tickers_allowed]
    value = current_ticker if current_ticker in tickers_allowed else tickers_allowed[0]
    return opts, value

@app.callback(
    Output("ticker-graph", "figure"),
    Input("ticker-dd", "value"),
    Input("price-field-dd", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("ma-window", "value"),
)
def update_ticker_graph(ticker, price_field, start_date, end_date, ma_window):
    if not ticker or not price_field:
        return go.Figure()
    mask = (prices["Ticker"] == ticker)
    if start_date:
        mask &= (prices["date"] >= pd.to_datetime(start_date))
    if end_date:
        mask &= (prices["date"] <= pd.to_datetime(end_date))
    sub = prices.loc[mask].sort_values("date").copy()
    if sub.empty:
        fig = go.Figure(); fig.update_layout(title="Нет данных для выбранного фильтра")
        return fig
    if isinstance(ma_window, (int, float)) and ma_window and ma_window > 1 and price_field in sub.columns:
        sub["MA"] = sub[price_field].rolling(int(ma_window), min_periods=1).mean()
        sub.attrs["ma_window"] = int(ma_window)
    fig = make_ticker_detail_fig(sub, price_field)
    return fig

@app.callback(
    Output("ticker-dd", "value", allow_duplicate=True),
    Output("table-info", "children"),
    Input("clusters-table", "selected_rows"),
    State("clusters-table", "data"),
    prevent_initial_call=True
)
def pick_from_table(selected_rows, data_rows):
    """Клик по строке таблицы — переключаем детальный график на этот тикер."""
    if not selected_rows:
        return no_update, ""
    idx = selected_rows[0]
    try:
        row = data_rows[idx]
        t = row["Ticker"]
        return t, f"Выбрано из таблицы: {t} (Cluster {row['Cluster']})"
    except Exception:
        return no_update, ""

if __name__ == "__main__":
    # host="0.0.0.0" — чтобы открыть с другого устройства в сети
    app.run(debug=True)


In [ ]:
#перебор большего количесвта гиперпарметров, ещё не запускался

from __future__ import annotations
import os, glob
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


DATA_DIR            = "data"
ALL_CSV_PATH        = "data/prices_all.csv"
PRICE_COL_PREF      = "auto"
MIN_ROWS_PER_TICKER = 50

# Грид автоэнкодера
LATENT_DIM_LIST = [2, 3, 4]
HIDDEN1_LIST    = [128, 256]
HIDDEN2_LIST    = [128, 256]
EPOCHS_LIST     = [80, 120]
LR_LIST         = [1e-3, 5e-4]
BATCH_SIZE_LIST = [16]


RUNS = 6

# Кластеризация
ALGO_LIST        = ["agglo", "kmeans"]
N_CLUSTERS_LIST  = [4, 6, 8, 10]


GRID_LIMIT = None

# Выводы
OUT_GRID_RESULTS_CSV   = "grid_results.csv"
OUT_ASSIGN_BEST_CSV    = "cluster_grid_best_assignments.csv"
OUT_PDF_BEST           = "clusters_grid_best_report.pdf"
OUT_HTML_BEST          = "cluster_grid_best_table.html"
MAKE_PDF               = True


def safe_float(x: Any) -> float:
    """Приводит к float, устойчиво к numpy-скалярам и массивам."""
    try:
        item = getattr(x, "item", None)
        if callable(item):
            return float(item())
    except Exception:
        pass
    try:
        arr = np.asarray(x)
        if arr.size == 1:
            return float(arr.reshape(-1)[0])
        return float(np.mean(arr))
    except Exception:
        return float(x)

def pick_device() -> torch.device:
    if torch.cuda.is_available(): return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def load_prices(path_all_csv: str, data_dir: str) -> pd.DataFrame:
    if os.path.exists(path_all_csv):
        df = pd.read_csv(path_all_csv)
    else:
        parts = []
        for p in glob.glob(os.path.join(data_dir, "*.csv")):
            try:
                dfi = pd.read_csv(p)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit(f"Нет CSV с колонкой 'Ticker' в {data_dir}")
        df = pd.concat(parts, ignore_index=True)

    ren = {}
    for want in ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]:
        for c in df.columns:
            if c.strip().lower() == want.lower():
                ren[c] = want; break
    df = df.rename(columns=ren)
    if "date" not in df.columns or "Ticker" not in df.columns:
        raise SystemExit("Нужны колонки: date, Ticker (+ цены).")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date","Ticker"]).reset_index(drop=True)
    return df

def pick_price_col(df: pd.DataFrame, pref: str) -> str:
    if pref != "auto":
        if pref in df.columns:
            return pref
        raise SystemExit(f"PRICE_COL_PREF='{pref}' не найден в данных.")
    return "Adj Close" if "Adj Close" in df.columns else ("Close" if "Close" in df.columns else "Close")

def filter_tickers_start_at_global_min(df: pd.DataFrame) -> List[str]:
    gmin = df["date"].min()
    firsts = df.groupby("Ticker")["date"].min()
    keep = firsts[firsts == gmin].index.tolist()
    return keep

def build_matrix_full(df: pd.DataFrame, price_col: str, min_rows: int,
                      tickers_keep: List[str]) -> Tuple[np.ndarray, List[str], pd.DatetimeIndex, pd.DataFrame]:
    dff = df[df["Ticker"].isin(set(tickers_keep))].copy()
    wide = dff.pivot(index="date", columns="Ticker", values=price_col).sort_index()

    good_cols = [c for c in wide.columns if wide[c].notna().sum() >= min_rows]
    wide = wide[good_cols]
    if wide.shape[1] == 0:
        raise SystemExit("После фильтрации по min_rows не осталось тикеров.")

    if wide.isna().any().any():
        bad = wide.columns[wide.isna().any()].tolist()
        raise SystemExit(f"Есть пропуски у: {bad[:10]}{' ...' if len(bad)>10 else ''}.")

    wmin = wide.min(axis=0); wmax = wide.max(axis=0)
    denom = (wmax - wmin).replace(0, np.nan)
    scaled = (wide - wmin) / denom
    scaled = scaled.fillna(0.0)

    X = scaled.T.values.astype(np.float32)
    companies = list(scaled.columns)
    dates = scaled.index
    return X, companies, dates, scaled

class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, h1: int, h2: int, z: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Linear(h2, z), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z, h2), nn.ReLU(),
            nn.Linear(h2, h1), nn.ReLU(),
            nn.Linear(h1, input_dim), nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.encoder(x); y = self.decoder(z); return y

def train_ae(X: np.ndarray, zdim: int, h1: int, h2: int,
             epochs: int, batch: int, lr: float, seed: int,
             device: torch.device) -> Tuple[np.ndarray, float]:
    torch.manual_seed(seed); np.random.seed(seed)
    n, d = X.shape
    model = Autoencoder(d, h1, h2, zdim).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    X_tensor = torch.from_numpy(X)
    if device.type == "cuda":
        X_tensor = X_tensor.pin_memory()
        torch.backends.cudnn.benchmark = True

    loader = torch.utils.data.DataLoader(
        X_tensor, batch_size=batch, shuffle=True,
        pin_memory=(device.type=="cuda"), num_workers=0
    )

    scaler = GradScaler(device="cuda", enabled=(device.type=="cuda"))

    model.train()
    for _ in range(epochs):
        for batch_x in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=(device.type=="cuda")):
                recon = model(batch_x)
                loss = loss_fn(recon, batch_x)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()


    model.eval()
    with torch.no_grad():
        Z_parts = []
        for batch_x in torch.utils.data.DataLoader(X_tensor, batch_size=batch, shuffle=False):
            batch_x = batch_x.to(device, non_blocking=True)
            z = model.encoder(batch_x).cpu().numpy()
            Z_parts.append(z)
        Z = np.vstack(Z_parts)

        recon_err, count = 0.0, 0
        for batch_x in torch.utils.data.DataLoader(X_tensor, batch_size=batch, shuffle=False):
            bx = batch_x.to(device, non_blocking=True)
            r = model(bx).cpu().numpy()
            recon_err += float(np.sum((r - batch_x.numpy())**2))
            count += batch_x.shape[0]
        recon_mse = recon_err / (count * d)

    return Z, recon_mse



def cluster_and_score(Z: np.ndarray, algo: str, k: int, seed: int) -> Dict:
    if algo == "agglo":
        labels = AgglomerativeClustering(n_clusters=k).fit_predict(Z)
    elif algo == "kmeans":
        labels = KMeans(n_clusters=k, random_state=seed, n_init="auto").fit_predict(Z)
    else:
        raise ValueError("algo must be 'agglo' or 'kmeans'")

    if len(set(labels)) > 1 and Z.shape[0] > k:
        sil = silhouette_score(Z, labels)
        cal = calinski_harabasz_score(Z, labels)
        dav = davies_bouldin_score(Z, labels)
    else:
        sil, cal, dav = -np.inf, -np.inf, np.inf

    return {
        "labels": labels,
        "silhouette": safe_float(sil),
        "calinski":  safe_float(cal),
        "davies":    safe_float(dav),
    }

def score_tuple(res: Dict) -> tuple[float,float,float,float]:
    """Критерий выбора лучшего: max CH → max Silhouette → min Davies → min ReconMSE."""
    m = res["metrics"]
    return (safe_float(m["calinski"]),
            safe_float(m["silhouette"]),
            -safe_float(m["davies"]),
            -safe_float(res["recon_mse"]))



def save_pdf(pdf_path: str, best: dict, companies: List[str], scaled: pd.DataFrame, price_col: str):
    Z = best["Z"]; labels = best["labels"]; algo = best["algo"]; K = best["n_clusters"]
    zdim = best["latent_dim"]; run = best["run"]; m = best["metrics"]; h1=best["hidden1"]; h2=best["hidden2"]
    with PdfPages(pdf_path) as pdf:

        pca2 = PCA(n_components=2).fit_transform(Z)
        fig = plt.figure(figsize=(7.2, 6.2))
        sc = plt.scatter(pca2[:,0], pca2[:,1], c=labels, cmap="tab10", s=70, edgecolor="k", linewidths=0.3)
        plt.title(f"{algo.upper()}  K={K}  z={zdim}  h=({h1},{h2})  run={run}  "
                  f"CH={m['calinski']:.1f}  Sil={m['silhouette']:.3f}  DB={m['davies']:.3f}  ({price_col})")
        plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True, alpha=.35); plt.colorbar(sc, label="cluster")
        pdf.savefig(fig); plt.close(fig)


        for cl in sorted(set(labels)):
            idx = [i for i, lab in enumerate(labels) if lab == cl]
            show = idx[:min(12, len(idx))]
            if not show: continue
            cols = [companies[i] for i in show]
            fig2 = plt.figure(figsize=(12, 5))
            plt.plot(scaled.index, scaled[cols].values, linewidth=1.1)
            plt.title(f"{algo.upper()}  Cluster {cl}  (first {len(cols)} of {len(idx)})")
            plt.xlabel("Date"); plt.ylabel("Normalised price (0..1)"); plt.grid(True, alpha=.35)
            plt.tight_layout(); pdf.savefig(fig2); plt.close(fig2)


        cluster_map = pd.DataFrame({"Company": companies, "Cluster": labels}).sort_values(["Cluster","Company"])
        fig3, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off"); ax.set_title("Company → Cluster", fontsize=14, pad=18)
        tbl = ax.table(cellText=cluster_map.values, colLabels=cluster_map.columns, loc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.15)
        pdf.savefig(fig3); plt.close(fig3)

def save_interactive_table_html(path: str, cluster_map: pd.DataFrame, info: Dict[str,str]):
    table_html = cluster_map.to_html(index=False, classes="display compact", table_id="clusters")
    meta = "".join(f"<li><b>{k}:</b> {v}</li>" for k,v in info.items())
    html = f"""<!doctype html>
<html><head>
<meta charset="utf-8"/>
<title>Cluster assignments (best grid result)</title>
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css"/>
<style>body{{font-family:Arial, sans-serif; margin:18px}}</style>
</head><body>
<h2>Best configuration — Cluster assignments</h2>
<ul>{meta}</ul>
{table_html}
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
<script>
$(function(){{
  $('#clusters').DataTable({{pageLength:25, lengthMenu:[10,25,50,100,200], order:[[1,'asc']], stateSave:true}});
}});
</script>
</body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)

def main():
    # устройство
    device = pick_device()
    print("[INFO] Device:", device)
    if device.type == "cuda":
        print("[INFO] CUDA build:", torch.version.cuda)
        print("[INFO] GPU:", torch.cuda.get_device_name(0))


    df = load_prices(ALL_CSV_PATH, DATA_DIR)
    price_col = pick_price_col(df, PRICE_COL_PREF)
    keep = filter_tickers_start_at_global_min(df)
    if not keep:
        raise SystemExit("Не найдено тикеров, начинающихся с первой даты.")
    X, companies, dates, scaled = build_matrix_full(df, price_col, MIN_ROWS_PER_TICKER, keep)
    print(f"[INFO] Using price column: {price_col}")
    print(f"[INFO] Global first date: {df['date'].min().date()}")
    print(f"[INFO] Tickers starting at first date: {len(companies)}")
    print(f"[INFO] Matrix: companies={len(companies)}, time_points={X.shape[1]}")


    combos = []
    for z in LATENT_DIM_LIST:
        for h1 in HIDDEN1_LIST:
            for h2 in HIDDEN2_LIST:
                for ep in EPOCHS_LIST:
                    for lr in LR_LIST:
                        for bs in BATCH_SIZE_LIST:
                            for run in range(RUNS):
                                combos.append((z,h1,h2,ep,lr,bs,run))
    total_ae = len(combos)
    total = total_ae * len(ALGO_LIST) * len(N_CLUSTERS_LIST)
    if GRID_LIMIT is not None:
        print(f"[WARN] Ограничиваем перебор до первых {GRID_LIMIT} комбинаций (для отладки).")

    results = []
    pbar = tqdm(total= total if GRID_LIMIT is None else GRID_LIMIT, desc="grid")
    stop = False
    for (z,h1,h2,ep,lr,bs,run) in combos:

        Z, recon_mse = train_ae(X, z, h1, h2, ep, bs, lr, seed=run, device=device)

        for algo in ALGO_LIST:
            for k in N_CLUSTERS_LIST:
                metrics = cluster_and_score(Z, algo, k, seed=run)
                results.append({
                    "algo": algo, "n_clusters": k, "latent_dim": z, "run": run,
                    "hidden1": h1, "hidden2": h2, "epochs": ep, "lr": lr, "batch_size": bs,
                    "Z": Z, "labels": metrics["labels"], "metrics": metrics, "recon_mse": recon_mse
                })
                pbar.update(1)
                if GRID_LIMIT is not None and len(results) >= GRID_LIMIT:
                    stop = True; break
            if stop: break
        if stop: break
    pbar.close()

    if not results:
        raise SystemExit("Нет результатов перебора (вероятно, слишком жёсткий GRID_LIMIT).")


    rows = []
    for r in results:
        m = r["metrics"]
        rows.append({
            "algo": r["algo"], "K": r["n_clusters"], "z": r["latent_dim"], "run": r["run"],
            "h1": r["hidden1"], "h2": r["hidden2"], "epochs": r["epochs"], "lr": r["lr"], "batch": r["batch_size"],
            "CH": safe_float(m["calinski"]), "Silhouette": safe_float(m["silhouette"]),
            "Davies": safe_float(m["davies"]), "ReconMSE": safe_float(r["recon_mse"])
        })
    grid_df = pd.DataFrame(rows)
    grid_df.sort_values(["CH","Silhouette","Davies"], ascending=[False,False,True], inplace=True)
    grid_df.to_csv(OUT_GRID_RESULTS_CSV, index=False)
    print(f"[INFO] Grid results saved → {OUT_GRID_RESULTS_CSV}")


    best = max(results, key=score_tuple)
    print("\n=== ЛУЧШАЯ КОНФИГУРАЦИЯ ===")
    print(f"algo={best['algo']}, K={best['n_clusters']}, z={best['latent_dim']}, run={best['run']}, "
          f"h1={best['hidden1']}, h2={best['hidden2']}, epochs={best['epochs']}, lr={best['lr']}, batch={best['batch_size']}")
    for k, v in best["metrics"].items():
        print(f"{k}: {safe_float(v):.6f}")
    print(f"Recon MSE: {safe_float(best['recon_mse']):.6f}")


    cluster_map = (
        pd.DataFrame({"Company": companies, "Cluster": best["labels"]})
        .sort_values(["Cluster","Company"]).reset_index(drop=True)
    )
    cluster_map.to_csv(OUT_ASSIGN_BEST_CSV, index=False)
    print(f"[INFO] Saved cluster assignments → {OUT_ASSIGN_BEST_CSV}")

    if MAKE_PDF:
        save_pdf(OUT_PDF_BEST, best, companies, scaled, price_col)
        print(f"[INFO] PDF report saved → {OUT_PDF_BEST}")

    info = {
        "Algorithm": best["algo"].upper(),
        "Clusters (K)": str(best["n_clusters"]),
        "Latent dim": str(best["latent_dim"]),
        "Run (seed)": str(best["run"]),
        "Hidden": f"{best['hidden1']},{best['hidden2']}",
        "Epochs": str(best["epochs"]),
        "LR": str(best["lr"]),
        "Batch": str(best["batch_size"]),
        "Calinski-Harabasz": f"{safe_float(best['metrics']['calinski']):.4f}",
        "Silhouette": f"{safe_float(best['metrics']['silhouette']):.4f}",
        "Davies-Bouldin": f"{safe_float(best['metrics']['davies']):.4f}",
        "Reconstruction MSE": f"{safe_float(best['recon_mse']):.6f}",
        "Price column": price_col,
        "Companies": str(len(companies)),
        "Time points": str(X.shape[1]),
        "Global first date": str(df['date'].min().date()),
    }
    save_interactive_table_html(OUT_HTML_BEST, cluster_map, info)
    print(f"[INFO] Interactive table saved → {OUT_HTML_BEST}")

if __name__ == "__main__":
    main()


In [5]:
# Спирмен и Пирсон

from __future__ import annotations
import os, glob
from typing import List, Tuple
import numpy as np
import pandas as pd
from pathlib import Path

import plotly.express as px
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


DATA_DIR      = "data"
ALL_CSV       = os.path.join(DATA_DIR, "prices_all.csv")
PRICE_COL     = "auto"
RET_KIND      = "log"
RESAMPLE      = None
START_DATE    = None
END_DATE      = None

OUT_DIR       = "corr"
TOP_N_PAIRS   = 30
PDF_FILE      = "correlations_report.pdf"


def load_prices() -> pd.DataFrame:
    if os.path.exists(ALL_CSV):
        df = pd.read_csv(ALL_CSV)
    else:
        parts: List[pd.DataFrame] = []
        for p in glob.glob(os.path.join(DATA_DIR, "*.csv")):
            try:
                dfi = pd.read_csv(p)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit("Не найдено ни одного CSV с колонкой 'Ticker' в ./data/")
        df = pd.concat(parts, ignore_index=True)

    ren = {}
    for want in ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]:
        for c in df.columns:
            if c.strip().lower() == want.lower():
                ren[c] = want
                break
    df = df.rename(columns=ren)
    if "date" not in df.columns or "Ticker" not in df.columns:
        raise SystemExit("В данных должны быть 'date' и 'Ticker'.")

    df["date"] = pd.to_datetime(df["date"])
    if START_DATE: df = df[df["date"] >= pd.to_datetime(START_DATE)]
    if END_DATE:   df = df[df["date"] <= pd.to_datetime(END_DATE)]
    df = df.sort_values(["date","Ticker"]).reset_index(drop=True)
    return df

def pick_price_col(df: pd.DataFrame) -> str:
    if PRICE_COL != "auto":
        if PRICE_COL in df.columns: return PRICE_COL
        raise SystemExit(f"PRICE_COL='{PRICE_COL}' нет в данных.")
    return "Adj Close" if "Adj Close" in df.columns else ("Close" if "Close" in df.columns else "Close")


def make_returns(df: pd.DataFrame, price_col: str) -> pd.DataFrame:
    wide = df.pivot(index="date", columns="Ticker", values=price_col).sort_index()
    if RESAMPLE:
        wide = wide.resample(RESAMPLE).last()

    wide = wide.dropna(how="any")
    if wide.shape[0] < 3 or wide.shape[1] < 2:
        raise SystemExit("Слишком мало данных после выравнивания по датам.")
    if RET_KIND == "log":
        rets = np.log(wide).diff().dropna(how="any")
    else:
        rets = wide.pct_change().dropna(how="any")
    return rets


def reorder_by_clustering(corr: pd.DataFrame) -> List[str]:
    dist = np.clip(1 - corr.values, 0, 2)
    Z = linkage(dist, method="average")
    order = leaves_list(Z)
    labels = corr.index.to_list()
    return [labels[i] for i in order], Z


def save_heatmaps_html(path: str, corr_p_ord: pd.DataFrame, corr_s_ord: pd.DataFrame):
    tabs = []
    for title, C in [("Pearson", corr_p_ord), ("Spearman", corr_s_ord)]:
        fig = px.imshow(
            C, x=C.columns, y=C.index, zmin=-1, zmax=1,
            color_continuous_scale="RdBu_r", aspect="auto", origin="lower",
            title=f"{title} correlation (ordered)"
        )
        fig.update_layout(margin=dict(l=40,r=20,t=60,b=40))
        tabs.append(fig.to_html(full_html=False, include_plotlyjs=False))

    html = f"""
<!doctype html>
<html><head>
<meta charset="utf-8"/>
<title>Stocks correlations</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{font-family:Arial,sans-serif;margin:16px}}
.tabs{{display:flex;gap:8px;margin-bottom:8px}}
.tabbtn{{padding:6px 10px;border:1px solid #ccc;border-radius:6px;cursor:pointer}}
.tabbtn.active{{background:#eee}}
.panel{{display:none}}
.panel.active{{display:block}}
</style>
</head><body>
<h2>Stocks correlations (returns): Pearson & Spearman</h2>
<div class="tabs">
  <button class="tabbtn active" onclick="showTab(0)">Pearson</button>
  <button class="tabbtn" onclick="showTab(1)">Spearman</button>
</div>
<div id="panels">
  <div class="panel active">{tabs[0]}</div>
  <div class="panel">{tabs[1]}</div>
</div>
<script>
function showTab(i){{
  const btns = document.querySelectorAll('.tabbtn');
  const pans = document.querySelectorAll('.panel');
  btns.forEach((b,idx)=>{{b.classList.toggle('active', idx===i)}});
  pans.forEach((p,idx)=>{{p.classList.toggle('active', idx===i)}});
}}
</script>
</body></html>
"""
    Path(path).write_text(html, encoding="utf-8")


def top_pairs(corr: pd.DataFrame, n=20) -> pd.DataFrame:

    C = corr.copy().rename_axis(index="A", columns="B")


    np.fill_diagonal(C.values, np.nan)


    abs_pairs = (
        C.abs()
         .stack()
         .rename("abs_corr")
         .reset_index()
         .dropna()
    )


    signed = (
        corr.rename_axis(index="A", columns="B")
            .stack()
            .rename("corr")
            .reset_index()
    )


    pairs = (
        abs_pairs.merge(signed, on=["A", "B"])
                 .loc[lambda df: df["A"] < df["B"]]
                 .sort_values("abs_corr", ascending=False)
                 .head(n)
                 .reset_index(drop=True)
    )
    return pairs


def save_pdf_report(pdf_path: str,
                    corr_p: pd.DataFrame, corr_s: pd.DataFrame,
                    corr_p_ord: pd.DataFrame, corr_s_ord: pd.DataFrame,
                    Zp, Zs,
                    topP: pd.DataFrame, topS: pd.DataFrame,
                    meta: dict):
    with PdfPages(pdf_path) as pdf:

        fig = plt.figure(figsize=(8.27, 11.69))
        ax = fig.add_axes([0.08, 0.08, 0.84, 0.84])
        ax.axis("off")
        ax.set_title("Stocks correlations — Report", fontsize=16, pad=12)
        lines = [
            f"Period: {meta['period']}",
            f"Frequency: {meta['freq']}, Returns: {meta['ret_kind']}",
            f"Tickers: {meta['n_tickers']}, Dates: {meta['n_dates']}",
        ]
        for i, line in enumerate(lines):
            ax.text(0.02, 0.95 - 0.06*i, line, fontsize=12, va="top")
        pdf.savefig(fig); plt.close(fig)


        fig = plt.figure(figsize=(11.69, 8.27))
        plt.title("Hierarchical clustering dendrogram (Pearson, distance = 1 - corr)")
        dendrogram(Zp, labels=corr_p.index.tolist(), leaf_rotation=90., leaf_font_size=6, color_threshold=None)
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)


        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        im = ax.imshow(corr_p_ord.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto", origin="lower")
        ax.set_title("Pearson correlation (ordered)")
        ax.set_xticks(range(corr_p_ord.shape[1])); ax.set_xticklabels(corr_p_ord.columns, rotation=90, fontsize=6)
        ax.set_yticks(range(corr_p_ord.shape[0])); ax.set_yticklabels(corr_p_ord.index, fontsize=6)
        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cbar.ax.set_ylabel("corr", rotation=270, labelpad=10)
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)


        fig = plt.figure(figsize=(11.69, 8.27))
        plt.title("Hierarchical clustering dendrogram (Spearman, distance = 1 - corr)")
        dendrogram(Zs, labels=corr_s.index.tolist(), leaf_rotation=90., leaf_font_size=6, color_threshold=None)
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)


        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        im = ax.imshow(corr_s_ord.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto", origin="lower")
        ax.set_title("Spearman correlation (ordered)")
        ax.set_xticks(range(corr_s_ord.shape[1])); ax.set_xticklabels(corr_s_ord.columns, rotation=90, fontsize=6)
        ax.set_yticks(range(corr_s_ord.shape[0])); ax.set_yticklabels(corr_s_ord.index, fontsize=6)
        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cbar.ax.set_ylabel("corr", rotation=270, labelpad=10)
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)


        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off"); ax.set_title(f"Top {len(topP)} pairs by |Pearson|", fontsize=14, pad=12)
        tbl = topP.copy()
        tbl["abs_corr"] = tbl["abs_corr"].round(4)
        tbl["corr"] = tbl["corr"].round(4)
        table = ax.table(cellText=tbl.values, colLabels=tbl.columns, loc="center")
        table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1, 1.2)
        pdf.savefig(fig); plt.close(fig)


        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off"); ax.set_title(f"Top {len(topS)} pairs by |Spearman|", fontsize=14, pad=12)
        tbl = topS.copy()
        tbl["abs_corr"] = tbl["abs_corr"].round(4)
        tbl["corr"] = tbl["corr"].round(4)
        table = ax.table(cellText=tbl.values, colLabels=tbl.columns, loc="center")
        table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1, 1.2)
        pdf.savefig(fig); plt.close(fig)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    df = load_prices()
    price_col = pick_price_col(df)
    print(f"[INFO] Price column: {price_col}")
    print(f"[INFO] Period: {df['date'].min().date()} .. {df['date'].max().date()}")

    rets = make_returns(df, price_col)
    print(f"[INFO] Returns shape: dates={rets.shape[0]} × tickers={rets.shape[1]} (freq={RESAMPLE or 'daily'}, kind={RET_KIND})")


    corr_pearson  = rets.corr(method="pearson")
    corr_spearman = rets.corr(method="spearman")


    orderP, Zp = reorder_by_clustering(corr_pearson)
    orderS, Zs = reorder_by_clustering(corr_spearman)
    corr_pearson_ord  = corr_pearson.loc[orderP, orderP]
    corr_spearman_ord = corr_spearman.loc[orderS, orderS]


    corr_pearson.to_csv(os.path.join(OUT_DIR, "corr_pearson_raw.csv"))
    corr_spearman.to_csv(os.path.join(OUT_DIR, "corr_spearman_raw.csv"))
    corr_pearson_ord.to_csv(os.path.join(OUT_DIR, "corr_pearson_ordered.csv"))
    corr_spearman_ord.to_csv(os.path.join(OUT_DIR, "corr_spearman_ordered.csv"))
    print(f"[INFO] Saved CSVs to ./{OUT_DIR}/")


    topP = top_pairs(corr_pearson, n=TOP_N_PAIRS)
    topS = top_pairs(corr_spearman, n=TOP_N_PAIRS)
    topP.to_csv(os.path.join(OUT_DIR, "top_pairs_pearson.csv"), index=False)
    topS.to_csv(os.path.join(OUT_DIR, "top_pairs_spearman.csv"), index=False)

    print("\n[Top pairs by |Pearson|]")
    print(topP.to_string(index=False, max_rows=TOP_N_PAIRS))
    print("\n[Top pairs by |Spearman|]")
    print(topS.to_string(index=False, max_rows=TOP_N_PAIRS))


    out_html = os.path.join(OUT_DIR, "correlations_heatmaps.html")
    save_heatmaps_html(out_html, corr_pearson_ord, corr_spearman_ord)
    print(f"[INFO] Interactive heatmaps → {out_html}")


    meta = {
        "period": f"{rets.index.min().date()} .. {rets.index.max().date()}",
        "freq": RESAMPLE or "daily",
        "ret_kind": RET_KIND,
        "n_tickers": rets.shape[1],
        "n_dates": rets.shape[0],
    }
    pdf_path = os.path.join(OUT_DIR, PDF_FILE)
    save_pdf_report(pdf_path,
                    corr_pearson, corr_spearman,
                    corr_pearson_ord, corr_spearman_ord,
                    Zp, Zs,
                    topP, topS,
                    meta)
    print(f"[INFO] PDF saved → {pdf_path}")

if __name__ == "__main__":
    main()


[INFO] Price column: Close
[INFO] Period: 2008-01-02 .. 2025-10-24
[INFO] Returns shape: dates=145 × tickers=200 (freq=daily, kind=log)


C:\Users\allll\AppData\Local\Temp\ipykernel_18512\2143057908.py:88: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



[INFO] Saved CSVs to ./corr/

[Top pairs by |Pearson|]
   A    B  abs_corr     corr
  HD  LOW  0.936650 0.936650
 COP  EOG  0.931066 0.931066
 PNC  TFC  0.931057 0.931057
  MA    V  0.928953 0.928953
 PNC  USB  0.925390 0.925390
  GS   MS  0.921346 0.921346
KLAC LRCX  0.920227 0.920227
 MCO SPGI  0.917477 0.917477
  BX  KKR  0.912365 0.912365
 TFC  USB  0.907774 0.907774
AMAT KLAC  0.905641 0.905641
 HLT  MAR  0.898705 0.898705
AMAT LRCX  0.897748 0.897748
 APO  KKR  0.895775 0.895775
 CVX  XOM  0.894714 0.894714
 COP  CVX  0.892868 0.892868
 COP  XOM  0.887281 0.887281
 JPM   MS  0.886345 0.886345
 AXP  TFC  0.883413 0.883413
  GS  JPM  0.878395 0.878395
 DUK   SO  0.878336 0.878336
 DHR  TMO  0.876409 0.876409
 BAC   MS  0.873173 0.873173
 COF  TFC  0.872655 0.872655
 EOG  XOM  0.871164 0.871164
 BAC    C  0.869009 0.869009
 BAC  WFC  0.867754 0.867754
 CVX  EOG  0.867277 0.867277
 AXP  COF  0.867041 0.867041
   C   MS  0.865540 0.865540

[Top pairs by |Spearman|]
   A    B  abs_corr

In [7]:
from __future__ import annotations
import os
import numpy as np
import pandas as pd

PEARSON_PATH  = "corr/corr_pearson_raw.csv"
SPEARMAN_PATH = "corr/corr_spearman_raw.csv"

def load_corr(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise SystemExit(f"Файл не найден: {path}")
    df = pd.read_csv(path, index_col=0)

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    df = df.loc[df.index, df.index]
    return df.astype(float)

def stats_abs(C: pd.DataFrame) -> tuple[float, float]:
    n = C.shape[0]
    iu = np.triu_indices(n, k=1)
    vals = np.abs(C.values[iu]).astype(float)
    return float(np.nanmean(vals)), float(np.nanmedian(vals))

def main():
    corr_p = load_corr(PEARSON_PATH)
    corr_s = load_corr(SPEARMAN_PATH)

    mean_p, med_p = stats_abs(corr_p)
    mean_s, med_s = stats_abs(corr_s)

    print(f"Mean |Pearson|   = {mean_p:.6f}")
    print(f"Median |Pearson| = {med_p:.6f}")
    print(f"Mean |Spearman|  = {mean_s:.6f}")
    print(f"Median |Spearman|= {med_s:.6f}")

if __name__ == "__main__":
    main()

Mean |Pearson|   = 0.344965
Median |Pearson| = 0.347163
Mean |Spearman|  = 0.258979
Median |Spearman|= 0.253299
